<div dir="rtl">

# الأسبوع 5: الشبكات العصبية CNN

---
**المُعد/المؤلف:** [عامر صوان](https://linkedin.com/in/amer-sawan) و [Mohammed Tattan]()

---

في الأسابيع الثلاثة الماضية أتقنّا أدوات الرؤية الكلاسيكية: وتعلمنا حول الخوارزميات (Canny للحواف، Gaussian للتنعيم، MOG2 للحركة). اليوم نتعلم كيف تختار الشبكة العصبية هذه الأدوات **بنفسها** من البيانات — وهذا هو جوهر **التعلم العميق**.

---


</div>

<div dir="rtl">

## 📋 محتويات هذا الدرس

1. 🖼️ **الصورة كمصفوفة أرقام** — Image as a Matrix of Numbers
2. ⚠️ **مشكلة تسطيح الصورة** — The Flattening Problem
3. ✨ **عملية الـ Convolution** — The Convolution Operation
4. 🌈 **الـ Kernels المتعددة وعمق الصورة** — Multiple Kernels & Image Depth
5. 🔗 **المسار الكامل للشبكة العصبية CNN** — Stacking CNN Layers
6. 🏊 **الـ Max Pooling**
7. 🧬 **لماذا تتفوق CNN على الشبكات العادية؟** — Inductive Biases
    - 7.1 🎯 الاتصال المحلي — Local Connectivity
    - 7.2 ↔️ التحول المتساوي — Translation Equivariance
    - 7.3 ♻️ مشاركة البارمترات — Parameter Sharing
    - 7.4 🐱 ثبات الترجمة — Translation Invariance
    - 7.5 🏔️ التسلسل الهرمي للميزات — Hierarchical Features
8. 📊 **مقارنة CNN مقابل Fully Connected**
9. 📚 مصادر للاستزادة

</div>

<div dir="rtl">

---

## 🖼️ 1. الصورة كمصفوفة أرقام | Image as a Matrix of Numbers

> **المفهوم الأساسي:** أي صورة رقمية هي في الحقيقة مجرد **مصفوفة من الأرقام**.
> كل رقم يمثل قيمة البكسل  — من **0** (أبيض) إلى **255** (أسود).

انظر للمثال التالي لصورة بحجم  **8×8 بكسل**:
- الأجزاء البيضاء = **0**
- الأجزاء الداكنة = **127 إلى 255**

<div align="center" style="display:flex;justify-content: center;">
<img src="./media/00-pixel-grid-data-matrix.png">
</div>


<div dir="rtl">

---

## ⚠️ 2. مشكلة تسطيح الصورة | The Flattening Problem

إن تسطيح الصورة لتصبح مصفوفة أحادية البعد يدمر المعلومات المكانية للبكسلات (Spatial Structure)

### لماذا التسطيح مشكلة؟
- بكسلان متجاوران رأسياً في الصورة يصبحان **بعيدَين** في المصفوفة الأحادية.
- الشبكة العادية **لا تعرف** أنهما كانا جارَين.
- المعلومات المحلية (Local Context) تضيع تماماً!


<div align="center" style="display:flex;justify-content: center;">
<img src="./media/01-flatting-problem.gif">
</div>

**الحل:** نحتاج عملية تحافظ على البنية المكانية — وهنا يأتي **Convolution**.

<div dir="rtl">

---

## ✨ 3. عملية الـ Convolution | The Convolution Operation

> نأخذ **Kernel صغيرة (3×3)** ونُمررها عبر الصورة.

### كيف تعمل؟
1. الـ Kernel يجلس فوق منطقة من الصورة.
2. **نضرب** كل قيمة في الـ kernel بقيمة البكسل المقابل.
3. **نجمع** جميع الحاصلات → رقم واحد في الـ output.
4. ننتقل للموضع التالي ونكرر.

### معادلة حجم الـ Output:
$$\text{Output} = (H - K + 1) \times (W - K + 1)$$

**مثال:**

- صورة **6×6**

- kernel **3×3** 

- output **4×4**


In [ ]:
from IPython.display import HTML
from base64 import b64decode
html_content_encoded = "CjxzdHlsZT4KICAgIC5nY2VsbCB7CiAgICAgICAgd2lkdGg6IDQ0cHg7CiAgICAgICAgaGVpZ2h0OiA0NHB4OwogICAgICAgIGJvcmRlci1yYWRpdXM6IDVweDsKICAgICAgICBkaXNwbGF5OiBmbGV4OwogICAgICAgIGFsaWduLWl0ZW1zOiBjZW50ZXI7CiAgICAgICAganVzdGlmeS1jb250ZW50OiBjZW50ZXI7CiAgICAgICAgZm9udC1zaXplOiAuOGVtOwogICAgICAgIGZvbnQtd2VpZ2h0OiBib2xkOwogICAgICAgIHRyYW5zaXRpb246IGFsbCAuM3M7CiAgICAgICAgYm9yZGVyOiAycHggc29saWQgdHJhbnNwYXJlbnQ7CiAgICB9CgogICAgLmljIHsKICAgICAgICBiYWNrZ3JvdW5kOiAjMWUzYTVmOwogICAgICAgIGNvbG9yOiAjOTNjNWZkOwogICAgfQoKICAgIC5pYy5obCB7CiAgICAgICAgYmFja2dyb3VuZDogIzNiODJmNjsKICAgICAgICBib3JkZXItY29sb3I6ICNhNzhiZmE7CiAgICAgICAgY29sb3I6IHdoaXRlOwogICAgICAgIHRyYW5zZm9ybTogc2NhbGUoMS4wNSk7CiAgICB9CgogICAgLmtjIHsKICAgICAgICBiYWNrZ3JvdW5kOiAjNDUxYTAzOwogICAgICAgIGNvbG9yOiAjZmJiZjI0OwogICAgICAgIGJvcmRlcjogMXB4IHNvbGlkICM3ODM1MGY7CiAgICB9CgogICAgLm9jIHsKICAgICAgICBiYWNrZ3JvdW5kOiAjMDY0ZTNiOwogICAgICAgIGNvbG9yOiAjNmVlN2I3OwogICAgICAgIGJvcmRlcjogMXB4IHNvbGlkICMwNjVmNDY7CiAgICB9CgogICAgLm9jLmZpbGxlZCB7CiAgICAgICAgYmFja2dyb3VuZDogIzA1OTY2OTsKICAgICAgICBjb2xvcjogd2hpdGU7CiAgICB9CgogICAgQGtleWZyYW1lcyBwb3AgewogICAgICAgIGZyb20gewogICAgICAgICAgICB0cmFuc2Zvcm06IHNjYWxlKC41KQogICAgICAgIH0KCiAgICAgICAgdG8gewogICAgICAgICAgICB0cmFuc2Zvcm06IHNjYWxlKDEpCiAgICAgICAgfQogICAgfQo8L3N0eWxlPgo8ZGl2IHN0eWxlPSJmb250LWZhbWlseTpTZWdvZSBVSSxzYW5zLXNlcmlmO2JhY2tncm91bmQ6IzBmMTcyYTtwYWRkaW5nOjMwcHg7Ym9yZGVyLXJhZGl1czoxNnB4O2NvbG9yOndoaXRlOyI+CiAgICA8aDMgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyO2NvbG9yOiNhNzhiZmE7bWFyZ2luLWJvdHRvbToyMHB4OyI+4pyoIENvbnZvbHV0aW9uIE9wZXJhdGlvbiDigJQg2KrZgdin2LnZhNmK2Kk8L2gzPgogICAgPGRpdiBzdHlsZT0iZGlzcGxheTpmbGV4O2p1c3RpZnktY29udGVudDpjZW50ZXI7YWxpZ24taXRlbXM6ZmxleC1zdGFydDtnYXA6MjBweDtmbGV4LXdyYXA6d3JhcDsiPgogICAgICAgIDxkaXYgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyOyI+CiAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojOTRhM2I4O21hcmdpbi1ib3R0b206OHB4OyI+SW5wdXQgKDbDlzYpPC9wPgogICAgICAgICAgICA8ZGl2IGlkPSJpZyIgc3R5bGU9ImRpc3BsYXk6Z3JpZDtncmlkLXRlbXBsYXRlLWNvbHVtbnM6cmVwZWF0KDYsNDRweCk7Z2FwOjJweDsiPjwvZGl2PgogICAgICAgIDwvZGl2PgogICAgICAgIDxkaXYgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyOyI+CiAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojZmJiZjI0O21hcmdpbi1ib3R0b206OHB4OyI+S2VybmVsICgzw5czKTwvcD4KICAgICAgICAgICAgPGRpdiBpZD0ia2ciIHN0eWxlPSJkaXNwbGF5OmdyaWQ7Z3JpZC10ZW1wbGF0ZS1jb2x1bW5zOnJlcGVhdCgzLDQ0cHgpO2dhcDoycHg7Ij48L2Rpdj4KICAgICAgICAgICAgPGRpdiBpZD0iY2QiCiAgICAgICAgICAgICAgICBzdHlsZT0ibWFyZ2luLXRvcDoxMHB4O3BhZGRpbmc6MTBweDtiYWNrZ3JvdW5kOiMxZTI5M2I7Ym9yZGVyLXJhZGl1czo4cHg7Zm9udC1zaXplOi44ZW07Y29sb3I6I2E3OGJmYTttaW4taGVpZ2h0OjUwcHg7bWluLXdpZHRoOjE1MHB4OyI+CiAgICAgICAgICAgIDwvZGl2PgogICAgICAgIDwvZGl2PgogICAgICAgIDxkaXYgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyOyI+CiAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojMzRkMzk5O21hcmdpbi1ib3R0b206OHB4OyI+T3V0cHV0ICg0w5c0KTwvcD4KICAgICAgICAgICAgPGRpdiBpZD0ib2ciIHN0eWxlPSJkaXNwbGF5OmdyaWQ7Z3JpZC10ZW1wbGF0ZS1jb2x1bW5zOnJlcGVhdCg0LDQ0cHgpO2dhcDoycHg7Ij48L2Rpdj4KICAgICAgICA8L2Rpdj4KICAgIDwvZGl2PgogICAgPGRpdiBzdHlsZT0idGV4dC1hbGlnbjpjZW50ZXI7bWFyZ2luLXRvcDoyMHB4O2Rpc3BsYXk6ZmxleDtnYXA6MTBweDtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO2ZsZXgtd3JhcDp3cmFwOyI+CiAgICAgICAgPGJ1dHRvbiBvbmNsaWNrPSJzdGVwQygpIgogICAgICAgICAgICBzdHlsZT0iYmFja2dyb3VuZDojNmQyOGQ5O2NvbG9yOndoaXRlO2JvcmRlcjpub25lO3BhZGRpbmc6MTBweCAyMnB4O2JvcmRlci1yYWRpdXM6OHB4O2N1cnNvcjpwb2ludGVyO2ZvbnQtc2l6ZToxZW07Ij7ilrYKICAgICAgICAgICAg2K7Yt9mI2KkgfCBTdGVwPC9idXR0b24+CiAgICAgICAgPGJ1dHRvbiBvbmNsaWNrPSJhdXRvQygpIgogICAgICAgICAgICBzdHlsZT0iYmFja2dyb3VuZDojMDU5NjY5O2NvbG9yOndoaXRlO2JvcmRlcjpub25lO3BhZGRpbmc6MTBweCAyMnB4O2JvcmRlci1yYWRpdXM6OHB4O2N1cnNvcjpwb2ludGVyO2ZvbnQtc2l6ZToxZW07Ij7ij6kKICAgICAgICAgICAg2KrZhNmC2KfYptmKIHwgQXV0bzwvYnV0dG9uPgogICAgICAgIDxidXR0b24gb25jbGljaz0icmVzZXRDKCkiCiAgICAgICAgICAgIHN0eWxlPSJiYWNrZ3JvdW5kOiMzNzQxNTE7Y29sb3I6d2hpdGU7Ym9yZGVyOm5vbmU7cGFkZGluZzoxMHB4IDIycHg7Ym9yZGVyLXJhZGl1czo4cHg7Y3Vyc29yOnBvaW50ZXI7Zm9udC1zaXplOjFlbTsiPuKGugogICAgICAgICAgICDYpdi52KfYr9ipPC9idXR0b24+CiAgICA8L2Rpdj4KPC9kaXY+CjxzY3JpcHQ+CiAgICBjb25zdCBpbnAgPSBbWzEsIDIsIDMsIDAsIDEsIDJdLCBbNCwgNSwgNiwgMSwgMCwgM10sIFs3LCA4LCA5LCAyLCAxLCA0XSwgWzAsIDEsIDIsIDMsIDQsIDVdLCBbMSwgMiwgMywgNCwgNSwgNl0sIFsyLCAzLCA0LCA1LCA2LCA3XV07CiAgICBjb25zdCBrZXIgPSBbWzEsIDAsIC0xXSwgWzEsIDAsIC0xXSwgWzEsIDAsIC0xXV07CiAgICBsZXQgc3QgPSAwLCBhdCA9IG51bGw7CiAgICBmdW5jdGlvbiBidWlsZCgpIHsKICAgICAgICBjb25zdCBpZyA9IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCJpZyIpLCBrZyA9IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCJrZyIpLCBvZyA9IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCJvZyIpOwogICAgICAgIGlnLmlubmVySFRNTCA9IGtnLmlubmVySFRNTCA9IG9nLmlubmVySFRNTCA9ICIiOwogICAgICAgIGlucC5mb3JFYWNoKChyb3csIHIpID0+IHJvdy5mb3JFYWNoKCh2LCBjKSA9PiB7IGNvbnN0IGQgPSBkb2N1bWVudC5jcmVhdGVFbGVtZW50KCJkaXYiKTsgZC5jbGFzc05hbWUgPSAiZ2NlbGwgaWMiOyBkLmlkID0gYGktJHtyfS0ke2N9YDsgZC50ZXh0Q29udGVudCA9IHY7IGlnLmFwcGVuZENoaWxkKGQpOyB9KSk7CiAgICAgICAga2VyLmZvckVhY2goKHJvdywgcikgPT4gcm93LmZvckVhY2goKHYsIGMpID0+IHsgY29uc3QgZCA9IGRvY3VtZW50LmNyZWF0ZUVsZW1lbnQoImRpdiIpOyBkLmNsYXNzTmFtZSA9ICJnY2VsbCBrYyI7IGQudGV4dENvbnRlbnQgPSB2OyBrZy5hcHBlbmRDaGlsZChkKTsgfSkpOwogICAgICAgIGZvciAobGV0IGkgPSAwOyBpIDwgMTY7IGkrKykgeyBjb25zdCBkID0gZG9jdW1lbnQuY3JlYXRlRWxlbWVudCgiZGl2Iik7IGQuY2xhc3NOYW1lID0gImdjZWxsIG9jIjsgZC5pZCA9IGBvLSR7TWF0aC5mbG9vcihpIC8gNCl9LSR7aSAlIDR9YDsgZC50ZXh0Q29udGVudCA9ICI/Ijsgb2cuYXBwZW5kQ2hpbGQoZCk7IH0KICAgIH0KICAgIGZ1bmN0aW9uIHN0ZXBDKCkgewogICAgICAgIGlmIChzdCA+PSAxNikgcmV0dXJuOwogICAgICAgIGNvbnN0IHIgPSBNYXRoLmZsb29yKHN0IC8gNCksIGMgPSBzdCAlIDQ7CiAgICAgICAgZG9jdW1lbnQucXVlcnlTZWxlY3RvckFsbCgiLmljIikuZm9yRWFjaChlID0+IGUuY2xhc3NMaXN0LnJlbW92ZSgiaGwiKSk7CiAgICAgICAgbGV0IHN1bSA9IDAsIHRlcm1zID0gW107CiAgICAgICAgZm9yIChsZXQga3IgPSAwOyBrciA8IDM7IGtyKyspZm9yIChsZXQga2MgPSAwOyBrYyA8IDM7IGtjKyspIHsKICAgICAgICAgICAgZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoYGktJHtyICsga3J9LSR7YyArIGtjfWApLmNsYXNzTGlzdC5hZGQoImhsIik7CiAgICAgICAgICAgIGNvbnN0IHAgPSBpbnBbciArIGtyXVtjICsga2NdICoga2VyW2tyXVtrY107IHN1bSArPSBwOwogICAgICAgICAgICBpZiAocCAhPT0gMCkgdGVybXMucHVzaChgJHtpbnBbciArIGtyXVtjICsga2NdfXgke2tlcltrcl1ba2NdfWApOwogICAgICAgIH0KICAgICAgICBkb2N1bWVudC5nZXRFbGVtZW50QnlJZCgiY2QiKS5pbm5lckhUTUwgPSBgPHNwYW4gc3R5bGU9ImNvbG9yOiNmYmJmMjQ7Ij7Yp9mE2K3Ys9in2Kg6PC9zcGFuPjxicj4ke3Rlcm1zLnNsaWNlKDAsIDQpLmpvaW4oIiArICIpfS4uLjxicj48c3BhbiBzdHlsZT0iY29sb3I6IzM0ZDM5OTsiPj0gJHtzdW19PC9zcGFuPmA7CiAgICAgICAgY29uc3Qgb2MgPSBkb2N1bWVudC5nZXRFbGVtZW50QnlJZChgby0ke3J9LSR7Y31gKTsgb2MudGV4dENvbnRlbnQgPSBzdW07IG9jLmNsYXNzTGlzdC5hZGQoImZpbGxlZCIpOyBvYy5zdHlsZS5hbmltYXRpb24gPSAicG9wIC4zcyBlYXNlIjsKICAgICAgICBzdCsrOwogICAgfQogICAgZnVuY3Rpb24gYXV0b0MoKSB7IGlmIChhdCkgeyBjbGVhckludGVydmFsKGF0KTsgYXQgPSBudWxsOyByZXR1cm47IH0gYXQgPSBzZXRJbnRlcnZhbCgoKSA9PiB7IGlmIChzdCA+PSAxNikgeyBjbGVhckludGVydmFsKGF0KTsgYXQgPSBudWxsOyByZXR1cm47IH0gc3RlcEMoKTsgfSwgNjAwKTsgfQogICAgZnVuY3Rpb24gcmVzZXRDKCkgeyBpZiAoYXQpIHsgY2xlYXJJbnRlcnZhbChhdCk7IGF0ID0gbnVsbDsgfSBzdCA9IDA7IGJ1aWxkKCk7IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCJjZCIpLmlubmVySFRNTCA9ICIiOyB9CiAgICBidWlsZCgpOwo8L3NjcmlwdD4="
HTML(b64decode(html_content_encoded).decode("utf-8"))

<div dir="rtl">

---

## 🌈 4. الـ Kernels المتعددة وعمق الصورة

### أولاً: لماذا نستخدم Kernels متعددة؟

كل **Kernel** مسؤولة عن اكتشاف **نمط واحد فقط** من الصورة.  
مثلاً: Kernel تكتشف الحواف الرأسية، وأخرى تكتشف الحواف الأفقية، وثالثة تكتشف الزوايا...

لذلك، نستخدم **عدة Kernels في نفس الوقت** على نفس الصورة، كل واحدة تُنتج خريطة ميزات (Feature Map) مستقلة.

$$\text{عدد الـ Output Channels} = \text{عدد الـ Kernels}$$




<div dir="rtl">

---

### ثانياً: فهم الأبعاد — ماذا تعني الرموز؟

لنبدأ بتعريف كل رمز بوضوح:

| الرمز | الاسم | المعنى |
|:-----:|:------|:-------|
| $H$ | Height — الارتفاع | عدد الصفوف (الأسطر) في الصورة أو الـ Feature Map |
| $W$ | Width — العرض | عدد الأعمدة في الصورة أو الـ Feature Map |
| $C$ | Channels (المدخل) | عمق الصورة المدخلة — كم طبقة فيها؟ |
| $K$ | Kernel Size — حجم الـ Kernel | طول وعرض نافذة الـ Kernel (مثلاً $3 \times 3$) |
| $C'$ | Channels (المخرج) | عدد الـ Feature Maps الناتجة = عدد الـ Kernels التي استخدمناها |
| $H'$ | Height الناتج | ارتفاع الـ Feature Map بعد تطبيق الـ Convolution |
| $W'$ | Width الناتج | عرض الـ Feature Map بعد تطبيق الـ Convolution |



<div dir="rtl">

---

### ثالثاً: حساب أبعاد الـ Feature Map

عندما نُمرِّر Kernel حجمها $K \times K$ على صورة حجمها $H \times W$، تصبح أبعاد الخرج الناتجة:

$$H' = H - K + 1$$

$$W' = W - K + 1$$

**مثال عددي:**
- صورة حجمها $6 \times 6$ أي $H=6,\ W=6$
- Kernel حجمها $3 \times 3$ أي $K=3$
- إذن: $H' = 6 - 3 + 1 = 4$ و $W' = 6 - 3 + 1 = 4$
- الخرج سيكون بحجم $4 \times 4$ ✅



<div dir="rtl">

---

### رابعاً: الصور الملونة — RGB Images

الصورة الرمادية (Grayscale) لها طبقة واحدة فقط، لكن الصورة الملونة لها **ثلاث طبقات**:

$$\underbrace{H \times W}_{\text{الأبعاد المكانية}} \times \underbrace{C}_{\text{عمق القناة}}$$

بالنسبة لصورة RGB:
$$C = 3 \quad \text{(Red, Green, Blue)}$$

حتى تتمكن الـ Kernel من معالجة الصورة الملونة، يجب أن يكون لها نفس عمق الصورة:

$$\text{حجم الـ Kernel} = K \times K \times C$$

في كل موضع، تضرب الـ Kernel كل قيمة في الـ $K \times K \times C$ عناصر وتجمعها → **رقم واحد فقط** كناتج.



<div dir="rtl">

---

### خامساً: الصورة الكاملة — من المدخل إلى المخرج

إذا استخدمنا $C'$ من الـ Kernels على صورة ملونة:

$$\underbrace{H \times W \times C}_{\text{المدخل}} \xrightarrow{\text{\ } C' \text{ Kernels\ }} \underbrace{H' \times W' \times C'}_{\text{المخرج}}$$

**تلخيص المعادلات الأساسية:**

$$H' = H - K + 1$$

$$W' = W - K + 1$$

$$C' = \text{عدد الـ Kernels}$$

---

### مثال شامل:

- **مدخل:** صورة ملونة $224 \times 224 \times 3$
- **نستخدم:** $32$ Kernel حجم كل منها $3 \times 3 \times 3$
- **الناتج:**

$$H' = 224 - 3 + 1 = 222$$
$$W' = 224 - 3 + 1 = 222$$
$$C' = 32$$

$$\text{الـ Output Volume} = 222 \times 222 \times 32$$

> 💡 **الفكرة الجوهرية:** كل Kernel تُنتج Feature Map مستقلة تكتشف نمطاً مختلفاً.  
> كلما زاد عدد الـ Kernels ($C'$)، زادت الأنماط التي تتعلمها الشبكة من الصورة.

</div>

<div dir="rtl">

---

## 🔗 5. تراكم طبقات CNN | Stacking CNN Layers

في شبكة CNN حقيقية، لا نكتفي بتطبيق Convolution مرة واحدة — بل نُكرِّر العملية عبر طبقات متعددة، حيث **ناتج كل طبقة يصبح مدخل الطبقة التالية** تلقائياً. هذا التراكم هو ما يمنح الشبكة قدرتها على فهم الصور بعمق.

---
في الشكل التالي مثال حول المسار الكامل للشبكة العصبية CNN
 
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/04-full-cnn-pipeline.png">
</div>

### ماذا يحدث لأبعاد البيانات مع كل طبقة؟

| الطبقة | الأبعاد المكانية ($H \times W$) | عدد الـ Channels ($C$) | ما الذي يحدث؟ |
|:-------|:-------------------------------|:----------------------|:--------------|
| **Input** | كبيرة — $224 \times 224$ | $3$ (RGB) | الصورة الأصلية الخام |
| **Conv 1** | تصغر — $112 \times 112$ | تزيد — $32$ | تكتشف حواف وتدرجات بسيطة |
| **Conv 2** | تصغر — $56 \times 56$ | تزيد — $64$ | تُركِّب الحواف في أشكال وزوايا |
| **Conv 3** | صغيرة جداً — $28 \times 28$ أو أقل | كثيرة جداً — $256$ أو $512$ | تتعرف على أجزاء وأنماط معقدة |
| **Flatten** | يتحول إلى مصفوفة أحادية البعد 1D | — | تحويل الـ Volume إلى قائمة أرقام |
| **FC + Softmax** | — | — | التصنيف النهائي |

---

### لماذا تصغر الأبعاد المكانية؟

في كل طبقة، تعمل عمليتان معاً على تقليص $H$ و $W$:

1. **الـ Convolution نفسها:** كما رأينا في المعادلة $H' = H - K + 1$، كل تطبيق للـ Kernel يُقلِّل الأبعاد قليلاً.
2. **الـ Max Pooling:** بعد كل طبقة Convolution عادةً، نُطبِّق Pooling يُقلِّص الأبعاد إلى النصف مرة واحدة — مثلاً $224 > 112 > 56 > 28$.

---

### لماذا يزيد عدد الـ Channels في نفس الوقت؟

عندما تصغر الأبعاد المكانية، نُعوِّض ذلك بزيادة عدد الـ Kernels في كل طبقة. أي أن:

$$\text{الطبقات الأولى: } H,W \text{ كبيرة} \quad C \text{ صغير}$$

$$\text{الطبقات العميقة: } H,W \text{ صغيرة} \quad C \text{ كبير}$$

هذا التبادل مقصود — الشبكة **تتنازل عن الدقة المكانية مقابل ثراء التمثيل (Feature Richness)**. بمعنى آخر: بدلاً من أن تعرف "أين بالضبط" كل بكسل، تصبح تعرف "ماذا يوجد" في الصورة بشكل أعمق.
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/05-cnn-depth-size.gif?a=a">
</div>



---

### ماذا يعني الـ Flatten؟

في نهاية طبقات الـ Convolution، لدينا حجم ثلاثي الأبعاد (Volume) بأبعاد صغيرة مثل $7 \times 7 \times 512$.  
نُحوِّله إلى متجه أحادي الأبعاد بطول:

$$7 \times 7 \times 512 = 25{,}088 \text{ رقم}$$

هذا المتجه هو **Feature Vector** — وهو ملخص الشبكة للصورة بأكملها. لا يحتوي على مواضع البكسلات، بل على **تقطير -خلاصة- لأهم الأنماط والميزات** التي تعلمتها الشبكة عبر كل طبقاتها، مضغوطةً في شكل مناسب لاتخاذ قرار التصنيف النهائي عبر الـ Fully Connected Layer.

</div>

<div dir="rtl">

---

## 🏊 6. الـ Max Pooling

**Max Pooling** هي عملية تقليص مقصودة تُقلِّص الأبعاد المكانية ($H$ و $W$) للـ Feature Map مع الاحتفاظ بأهم المعلومات فيها. تأتي عادةً بعد كل طبقة Convolution مباشرةً، وتُشكِّل معها ثنائياً أساسياً في بنية الـ CNN.

### كيف يعمل؟

1. نأخذ **نافذة $2 \times 2$** ونُمرِّرها على الـ Feature Map **بدون تداخل** (أي أن النافذة تقفز بمقدار 2 في كل خطوة — وهذا ما يُسمى الـ Stride).
2. من كل نافذة نأخذ **القيمة العظمى (Maximum)** فقط ونتجاهل الثلاث قيم الأخرى.
3. خريطة $4 \times 4$ تصبح $2 \times 2$ — أي تقليل الأبعاد بنسبة $75\%$ دفعةً واحدة!

بشكل عام، أي Feature Map بأبعاد $H \times W$ تصبح بعد تطبيق Max Pooling حجمه $2 \times 2$ كالتالي:

$$H' = \frac{H}{2} \qquad W' = \frac{W}{2}$$
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/06-maxpooling.gif">
</div>

### لماذا نأخذ الـ Maximum تحديداً؟

القيمة العظمى في النافذة تمثل **أقوى استجابة** للـ Kernel في تلك المنطقة — أي أنها تُجيب على سؤال: **"هل وُجد هذا النمط في هذه المنطقة؟"**. لا يهمنا أين بالضبط داخل النافذة، يكفي أنه موجود. بهذا نحتفظ بـ **وجود الميزة** ونتخلى عن **موضعها الدقيق**، وهذا بالضبط ما يمنح الشبكة خاصية الـ Translation Invariance التي سنشرحها لاحقا — أي أن الكائن يُتعرَّف عليه بصرف النظر عن مكانه في الصورة.

بخلاف Max Pooling، هناك نوع آخر يُسمى **Average Pooling** يأخذ متوسط القيم بدل الأعلى، لكن Max Pooling أثبت تفوقاً عملياً في معظم مهام رؤية الحاسوب لأن الميزات القوية في الصورة تُشفَّر في القيم العالية لا في المتوسطات.

</div>

<div dir="rtl">

---

## 🧬 7. لماذا تتفوق CNN على الشبكات العادية؟

تخيّل أنك تُريد تعليم طفل التعرف على القطط — لن تُريه كل بكسل في الصورة وتقول "احفظ هذا". بدلاً من ذلك ستُعلِّمه: "انظر للأذنين المدببتين، والعينين اللوزيتين، والفراء". هذا بالضبط ما تفعله الـ CNN — لكن بدلاً من أن نُخبرها بما تبحث عنه، **تتعلمه بنفسها من البيانات**.

الـ CNN لا تتفوق بالصدفة. بنيتها مصممة بناءً على **افتراضات ذكية مدمجة في معمارها** تعكس حقائق فيزيائية حقيقية عن طبيعة الصور. هذه الافتراضات تُسمى **Inductive Biases**، وهي التي تجعلها أكثر كفاءةً بما لا يُقاس من الشبكات العصبية العادية (Fully Connected Networks) عند التعامل مع البيانات البصرية — سواء كانت صوراً أو فيديوهات أو حتى أشعة طبية.


</div>


<div dir="rtl">


---

### 7.1. 🎯 Local Connectivity — الاتصال المحلي

كل نيورون في طبقة الـ Convolution **لا يرى سوى منطقة صغيرة محلية** من الصورة — فقط الـ $K \times K$ بكسل التي تغطيها الـ Kernel، ولا يعلم شيئاً عمّا يقع خارجها.

**لماذا هذا منطقي؟** لأن في الصور الحقيقية، البكسلات القريبة من بعضها هي التي تحمل معلومات مشتركة — حافة السيارة، ريشة الطائر، حرف في كلمة. أما بكسل في الركن الأيسر العلوي فلا علاقة له عموماً ببكسل في الركن الأيمن السفلي.

**مثال:** لو أردنا اكتشاف عيون الإنسان في صورة، لا نحتاج للنظر إلى القدمين — يكفي النظر إلى منطقة الوجه. الاتصال المحلي يجعل الشبكة تفعل ذلك تلقائياً.

**القوة:** في شبكة Fully Connected على صورة $224 \times 224 \times 3$، كل نيورون يرى $224 \times 224 \times 3 = 150{,}528$ قيمة. في CNN، كل نيورون يرى $3 \times 3 \times 3 = 27$ قيمة فقط — تقليل بمقدار $\times 5{,}575$ في عدد العمليات الحسابية لكل نيورون.

</div>
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/08-1local-connectivity.gif">
</div>



<div dir="rtl">
---


### 7.2. ↔️ Translation Equivariance — التحول المتساوي

إذا تحركت ميزة ما في الصورة المدخلة، **يتحرك الناتج في الـ Feature Map بنفس المقدار** تماماً. الـ Convolution لا تُفضِّل موضعاً على آخر — تُطبَّق بنفس الطريقة في كل مكان.

**مثال:** إذا كانت حافة رأسية تقع عند العمود الثالث في الصورة، فستظهر الاستجابة عند العمود الثالث في الـ Feature Map. لو تحركت الحافة إلى العمود السادس، تتبعها الاستجابة تلقائياً دون أي تدريب إضافي.

**القوة:** الشبكة تتعلم اكتشاف الحافة **مرة واحدة فقط**، وتُطبِّق هذا التعلم في كل موضع من الصورة — في أعلى اليمين، أسفل اليسار، في المنتصف، في كل مكان.

</div>
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/08-2-translation-eq.png">
</div>


<div dir="rtl">
---


### 7.3. ♻️ Parameter Sharing — مشاركة البارمترات


**نفس الـ Kernel بنفس الأوزان** تُمرَّر على كل مواضع الصورة. لا توجد Kernel مختلفة للركن الأيسر وأخرى للركن الأيمن — Kernel واحدة تفعل كل شيء.

**مثال:** Kernel مدرَّبة على اكتشاف الحواف الرأسية ستكتشفها سواء كانت الحافة في السماء أو على الأرض أو في المنتصف — نفس الأوزان، نفس القدرة، في كل موضع.

**القوة:** لو كان لدينا Kernel مستقلة لكل موضع في صورة $224 \times 224$، لاحتجنا إلى:
$$224 \times 224 \times (3 \times 3 \times 3) = 1{,}354{,}752 \text{ معامل لـ Kernel واحدة فقط!}$$
لكن بفضل Parameter Sharing، كل Kernel تحتاج فقط:
$$3 \times 3 \times 3 = 27 \text{ معامل}$$
هذا تقليل بمقدار $\times 50{,}000$ في عدد المعاملات — مما يجعل التدريب أسرع وأقل عرضة للـ Overfitting.

</div>
<div align="center" style="display:flex;justify-content: center;">
<img src="./media/08-3parameter-sharing.gif">
</div>




<div dir="rtl">
---


### 7.4. 🐱 Translation Invariance — ثبات الترجمة

بفضل الـ Max Pooling، لا يهم أين يقع الكائن في الصورة — **الشبكة ستُصنِّفه بنفس الدقة** سواء كان في الركن أو المنتصف أو أي مكان آخر.

**مثال:** صورة قطة في الجهة اليسرى وصورة نفس القطة في الجهة اليمنى — كلتاهما تحصل على نفس التصنيف "قطة". هذا لأن الـ Pooling تُدمِّر المعلومات الموضعية الدقيقة تدريجياً طبقةً بعد طبقة، محتفظةً فقط بـ "هل الميزة موجودة؟" لا "أين هي بالضبط؟"

**القوة:** بدون هذه الخاصية، كانت الشبكة ستحتاج لرؤية القطة في **كل موضع ممكن** من الصورة أثناء التدريب حتى تتعلم التعميم — وهذا يعني بيانات تدريب أكبر بأضعاف مضاعفة.

</div>

<div align="center" style="display:flex;justify-content: center;">
<img src="./media/08-4translation-inv.png">
</div>



<div dir="rtl">
---


### 7.5. 🏔️ Hierarchical Features — التسلسل الهرمي للميزات

الطبقات لا تتعلم كلها على نفس المستوى من التجريد — بل تبني فهمها **من البسيط إلى المعقد** تدريجياً:

$$\underbrace{\text{حواف وتدرجات}}_{\text{Layer 1}} \longrightarrow \underbrace{\text{أشكال وأنسجة}}_{\text{Layer 2}} \longrightarrow \underbrace{\text{أجزاء: عين، عجلة، ورقة}}_{\text{Layer 3}} \longrightarrow \underbrace{\text{كائنات كاملة: وجه، سيارة، شجرة}}_{\text{Layer 4+}}$$

**مثال:** لتعرُّف الشبكة على وجه إنسان — الطبقة الأولى تكتشف الحواف، الثانية تُركِّبها في أشكال بيضاوية، الثالثة تعرف أن هذه الأشكال قد تكون عيناً أو فماً، والطبقة الأعمق تُركِّب كل ذلك معاً لتقول: "هذا وجه إنسان".

**القوة:** هذا التسلسل يعني أن الطبقات الأولى **قابلة لإعادة الاستخدام** عبر مهام مختلفة — الحواف والأشكال البسيطة مفيدة سواء كنا نُصنِّف قطط أو سيارات أو أشعة طبية. هذا هو أساس فكرة الـ **Transfer Learning** التي سنتعلمها لاحقاً.


</div>

In [ ]:
from IPython.display import HTML
from base64 import b64decode
html_encoded = "PHN0eWxlPgogICAgLmhpIHsKICAgICAgICBkaXNwbGF5OiBmbGV4OwogICAgICAgIGFsaWduLWl0ZW1zOiBmbGV4LXN0YXJ0OwogICAgICAgIGdhcDogMjBweDsKICAgICAgICBwYWRkaW5nLWxlZnQ6IDM1cHg7CiAgICAgICAgbWFyZ2luLWJvdHRvbTogMThweDsKICAgIH0KCiAgICAuaGQgewogICAgICAgIHdpZHRoOiAyMnB4OwogICAgICAgIGhlaWdodDogMjJweDsKICAgICAgICBib3JkZXItcmFkaXVzOiA1MCU7CiAgICAgICAgYm9yZGVyOiAzcHggc29saWQ7CiAgICAgICAgZmxleC1zaHJpbms6IDA7CiAgICAgICAgbWFyZ2luLXRvcDogMTBweDsKICAgICAgICBib3gtc2hhZG93OiAwIDAgMTBweCByZ2JhKDI1NSwgMjU1LCAyNTUsIC4zKTsKICAgIH0KCiAgICAuaGIgewogICAgICAgIGZsZXg6IDE7CiAgICAgICAgcGFkZGluZzogMTRweCAxOHB4OwogICAgICAgIGJvcmRlci1yYWRpdXM6IDEwcHg7CiAgICAgICAgYm9yZGVyOiAxcHggc29saWQ7CiAgICAgICAgdHJhbnNpdGlvbjogdHJhbnNmb3JtIC4yczsKICAgIH0KCiAgICAuaGI6aG92ZXIgewogICAgICAgIHRyYW5zZm9ybTogdHJhbnNsYXRlWCg1cHgpOwogICAgfQoKICAgIEBrZXlmcmFtZXMgc1IgewogICAgICAgIGZyb20gewogICAgICAgICAgICBvcGFjaXR5OiAwOwogICAgICAgICAgICB0cmFuc2Zvcm06IHRyYW5zbGF0ZVgoLTMwMHB4KQogICAgICAgIH0KCiAgICAgICAgdG8gewogICAgICAgICAgICBvcGFjaXR5OiAxOwogICAgICAgICAgICB0cmFuc2Zvcm06IHRyYW5zbGF0ZVgoMCkKICAgICAgICB9CiAgICB9Cjwvc3R5bGU+CjxkaXYgZGlyPSJydGwiIHN0eWxlPSJmb250LWZhbWlseTpTZWdvZSBVSSxzYW5zLXNlcmlmO2JhY2tncm91bmQ6IzBmMTcyYTtwYWRkaW5nOjMwcHg7Ym9yZGVyLXJhZGl1czoxNnB4O2NvbG9yOndoaXRlOyI+CiAgICA8aDMgc3R5bGU9InRleHQtYWxpZ246Y2VudGVyO2NvbG9yOiNhNzhiZmE7bWFyZ2luLWJvdHRvbToyNXB4OyI+8J+PlO+4jyDYp9mE2KrYs9mE2LPZhCDYp9mE2YfYsdmF2Yog2YTZhNmF2YrYstin2Ko8L2gzPgogICAgPGRpdiBzdHlsZT0icG9zaXRpb246cmVsYXRpdmU7bWF4LXdpZHRoOjY4MHB4O21hcmdpbjowIGF1dG87Ij4KICAgICAgICA8ZGl2CiAgICAgICAgICAgIHN0eWxlPSJwb3NpdGlvbjphYnNvbHV0ZTtyaWdodDoxM3B4O3RvcDowO2JvdHRvbTowO3dpZHRoOjNweDtiYWNrZ3JvdW5kOmxpbmVhci1ncmFkaWVudCgxODBkZWcsIzNiODJmNiwjNmQyOGQ5LCNhNzhiZmEsI2VjNDg5OSk7Ym9yZGVyLXJhZGl1czozcHg7Ij4KICAgICAgICA8L2Rpdj4KCiAgICAgICAgPGRpdiBjbGFzcz0iaGkiIHN0eWxlPSJhbmltYXRpb246c1IgLjVzIGVhc2UgLjFzIGJvdGg7Ij4KICAgICAgICAgICAgPGRpdiBjbGFzcz0iaGQiIHN0eWxlPSJiYWNrZ3JvdW5kOiMzYjgyZjY7Ym9yZGVyLWNvbG9yOiM5M2M1ZmQ7Ij48L2Rpdj4KICAgICAgICAgICAgPGRpdiBjbGFzcz0iaGIiIHN0eWxlPSJib3JkZXItY29sb3I6IzNiODJmNjtiYWNrZ3JvdW5kOnJnYmEoNTksMTMwLDI0NiwuMSk7Ij4KICAgICAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojOTNjNWZkO2ZvbnQtd2VpZ2h0OmJvbGQ7bWFyZ2luOjA7Ij7Yp9mE2YXYs9iq2YjZiSDYp9mE2KPZiNmEIC0g2YXZitiy2KfYqiDYqNiz2YrYt9ipPC9wPgogICAgICAgICAgICAgICAgPHAgc3R5bGU9ImNvbG9yOiNjYmQ1ZTE7Zm9udC1zaXplOi45ZW07bWFyZ2luOjhweCAwIDA7Ij7YrdmI2KfZgSDYqNiz2YrYt9ip2Iwg2KrYr9ix2KzYp9iq2Iwg2K7Yt9mI2Lcg2LHYo9iz2YrYqSDZiNij2YHZgtmK2Kk8L3A+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6IzQ3NTU2OTtmb250LXNpemU6LjhlbTttYXJnaW46M3B4IDAgMDsiPtit2YjYp9mBINio2LPZiti32KnYjCDYstmI2KfZitin2Iwg2K7Yt9mI2Lcg2YXYp9im2YTYqTwvcD4KICAgICAgICAgICAgPC9kaXY+CiAgICAgICAgPC9kaXY+CgogICAgICAgIDxkaXYgY2xhc3M9ImhpIiBzdHlsZT0iYW5pbWF0aW9uOnNSIC41cyBlYXNlIC4zcyBib3RoOyI+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhkIiBzdHlsZT0iYmFja2dyb3VuZDojNmQyOGQ5O2JvcmRlci1jb2xvcjojYTc4YmZhOyI+PC9kaXY+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhiIiBzdHlsZT0iYm9yZGVyLWNvbG9yOiM2ZDI4ZDk7YmFja2dyb3VuZDpyZ2JhKDEwOSw0MCwyMTcsLjEpOyI+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6I2M0YjVmZDtmb250LXdlaWdodDpib2xkO21hcmdpbjowOyI+2KfZhNi32KjZgtipINin2YTYq9in2YbZitipIC0g2YXZitiy2KfYqiDZhdiq2YjYs9i32Kk8L3A+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6I2NiZDVlMTtmb250LXNpemU6LjllbTttYXJnaW46OHB4IDAgMDsiPtij2YbYs9is2KnYjCDYstmI2KfZitin2Iwg2KPYtNmD2KfZhCDZhdix2YPYqNipINmF2YYg2K3ZiNin2YE8L3A+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6IzQ3NTU2OTtmb250LXNpemU6LjhlbTttYXJnaW46M3B4IDAgMDsiPtiy2YjYp9mK2KfYjCDYo9i02YPYp9mEINmF2LHZg9io2Kkg2YXZhiDYrdmI2KfZgTwvcD4KICAgICAgICAgICAgPC9kaXY+CiAgICAgICAgPC9kaXY+CgogICAgICAgIDxkaXYgY2xhc3M9ImhpIiBzdHlsZT0iYW5pbWF0aW9uOnNSIC41cyBlYXNlIC41cyBib3RoOyI+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhkIiBzdHlsZT0iYmFja2dyb3VuZDojYTc4YmZhO2JvcmRlci1jb2xvcjojYzRiNWZkOyI+PC9kaXY+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhiIiBzdHlsZT0iYm9yZGVyLWNvbG9yOiNhNzhiZmE7YmFja2dyb3VuZDpyZ2JhKDE2NywxMzksMjUwLC4xKTsiPgogICAgICAgICAgICAgICAgPHAgc3R5bGU9ImNvbG9yOiNkZGQ2ZmU7Zm9udC13ZWlnaHQ6Ym9sZDttYXJnaW46MDsiPtin2YTZhdiz2KrZiNmJINin2YTYq9in2YTYqyAtINij2KzYstin2KEg2LnYp9mE2YrYqSDYp9mE2YXYs9iq2YjZiTwvcD4KICAgICAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojY2JkNWUxO2ZvbnQtc2l6ZTouOWVtO21hcmdpbjo4cHggMCAwOyI+2KPYrNiy2KfYoSDZgtin2KjZhNipINmE2YTYqti52LHZgTog2LnZitmGIPCfkYHvuI/YjCDYudis2YTYqSDwn5S12Iwg2YjYsdmC2Kkg2LTYrNixCiAgICAgICAgICAgICAgICAgICAg8J+NgzwvcD4KICAgICAgICAgICAgICAgIDxwIHN0eWxlPSJjb2xvcjojNDc1NTY5O2ZvbnQtc2l6ZTouOGVtO21hcmdpbjozcHggMCAwOyI+2KPYrNiy2KfYoSDZgtin2KjZhNipINmE2YTYqti52LHZgTog2LnZitmGIPCfkYHvuI/YjCDYudis2YTYqSDwn5S12Iwg2YjYsdmC2Kkg2LTYrNixCiAgICAgICAgICAgICAgICAgICAg8J+NgzwvcD4KICAgICAgICAgICAgPC9kaXY+CiAgICAgICAgPC9kaXY+CgogICAgICAgIDxkaXYgY2xhc3M9ImhpIiBzdHlsZT0iYW5pbWF0aW9uOnNSIC41cyBlYXNlIC43cyBib3RoOyI+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhkIiBzdHlsZT0iYmFja2dyb3VuZDojZWM0ODk5O2JvcmRlci1jb2xvcjojZjlhOGQ0OyI+PC9kaXY+CiAgICAgICAgICAgIDxkaXYgY2xhc3M9ImhiIiBzdHlsZT0iYm9yZGVyLWNvbG9yOiNlYzQ4OTk7YmFja2dyb3VuZDpyZ2JhKDIzNiw3MiwxNTMsLjEpOyI+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6I2Y5YThkNDtmb250LXdlaWdodDpib2xkO21hcmdpbjowOyI+2KfZhNi32KjZgtipINin2YTYsdin2KjYudipIC0g2YPYp9im2YbYp9iqINmD2KfZhdmE2Kk8L3A+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6I2NiZDVlMTtmb250LXNpemU6LjllbTttYXJnaW46OHB4IDAgMDsiPtmD2KfYptmG2KfYqiDZg9in2YXZhNipOiDZiNis2Ycg8J+YitiMINiz2YrYp9ix2Kkg8J+al9iMINi02KzYsdipIPCfjLM8L3A+CiAgICAgICAgICAgICAgICA8cCBzdHlsZT0iY29sb3I6IzQ3NTU2OTtmb250LXNpemU6LjhlbTttYXJnaW46M3B4IDAgMDsiPtmD2KfYptmG2KfYqiDZg9in2YXZhNipOiDZiNis2Ycg8J+YitiMINiz2YrYp9ix2Kkg8J+al9iMINi02KzYsdipIPCfjLM8L3A+CiAgICAgICAgICAgIDwvZGl2PgogICAgICAgIDwvZGl2PgogICAgPC9kaXY+CgogICAgPGRpdgogICAgICAgIHN0eWxlPSJtYXJnaW4tdG9wOjIwcHg7cGFkZGluZzoxNXB4O2JhY2tncm91bmQ6cmdiYSgxNjcsMTM5LDI1MCwuMSk7Ym9yZGVyLWxlZnQ6NHB4IHNvbGlkICNhNzhiZmE7Ym9yZGVyLXJhZGl1czo4cHg7bWF4LXdpZHRoOjY4MHB4O21hcmdpbjoyMHB4IGF1dG8gMDsiPgogICAgICAgIDxwIHN0eWxlPSJjb2xvcjojYzRiNWZkO21hcmdpbjowOyI+8J+UkSA8c3Ryb25nPtin2YTZhdio2K/YoyDYp9mE2KzZiNmH2LHZiiDZgdmKIERlZXAgTGVhcm5pbmc6PC9zdHJvbmc+INmD2YQg2LfYqNmC2Kkg2KrYqNmG2Yog2LnZhNmJINmF2KcKICAgICAgICAgICAg2KrYudmE2YXYqtmHINin2YTYs9in2KjZgtip2Iwg2YXZhdinINmK2YbYqtisINiq2YXYq9mK2YTYp9mLINmF2KrYstin2YrYryDYp9mE2KrYrNix2YrYryDZiNin2YTZgtmI2KkuPC9wPgogICAgPC9kaXY+CjwvZGl2Pg=="
html = b64decode(html_encoded).decode("utf-8")
HTML(html)

<div dir="rtl">

---


## 8. ملخص المقارنة: CNN مقابل Fully Connected

| الخاصية | Fully Connected | CNN |
|:--------|:---------------|:----|
| كل نيورون يرى | الصورة كلها | منطقة محلية صغيرة فقط |
| عدد المعاملات | ضخم جداً | صغير بفضل Parameter Sharing |
| التعامل مع الإزاحة | يجب إعادة التدريب | تلقائي بفضل Equivariance |
| التعامل مع الموضع | حساس جداً | متسامح بفضل Pooling |
| مستوى التجريد | طبقة واحدة مسطحة | هرمي متدرج |


💡 هذه الخصائص الخمس مجتمعةً هي السبب في أن CNN تستطيع تحقيق دقة تتجاوز $99\%$ في التعرف على الأرقام المكتوبة بخط اليد، وأكثر من $95\%$ في تصنيف آلاف الفئات من الصور — بينما ستفشل شبكة Fully Connected في تحقيق ذلك بنفس الكفاءة مهما كبر حجمها.


<div dir="rtl">

---
## 9. 📚 مصادر للاستزادة | Further Reading
</div>


- **Deep Learning Book** — Ian Goodfellow et al. (Chapter 9: Convolutional Networks)
- **CS231n** — Stanford Course on CNNs for Visual Recognition
- [**3Blue1Brown** — Neural Networks video series (YouTube)](https://www.youtube.com/c/3blue1brown)
- [Convolutional Neural Networks (CNNs) - Explained](https://www.youtube.com/watch?v=YGILT182T6w&t=82s)

---
*Computer Vision Course — Week 4 & 5*